# Модуль 4a — micrograd за час

Здесь ты **своими руками** соберёшь крошечный движок, который умеет то
же, что `loss.backward()` в PyTorch — только ~50 строк чистого Python
без единой библиотеки.

Полный гайд с картинками и пояснениями — в
[README этой папки](https://github.com/ITrubnikov/Train_of_Thought-homework/tree/main/notebooks/module-4a-micrograd).
Лекция — [Модуль 4a](https://itrubnikov.github.io/Train_of_Thought/docs/modules/04a-micrograd/).

Зависимостей нет, кроме `matplotlib` для финального графика (в Colab уже стоит).
Прогоняй сверху вниз. Места с `# TODO` — это домашка.

## Шаг 0 — посчитай руками (на бумаге, без кода)

```text
   a=2 --+
         +--(x)--> e=6 --+
   b=3 --+               +--(+)--> L=10
   c=4 ------------------+
```

Forward: `e = a*b = 6`, `L = e + c = 10`.

Backward (стартуем с `dL/dL = 1`):
- `+`: пропускает насквозь → `dL/de = 1`, `dL/dc = 1`
- `x`: отдаёт другой множитель → `dL/da = 1*b = 3`, `dL/db = 1*a = 2`

Запиши: **a.grad=3, b.grad=2, c.grad=1**. В Шаге 2 код должен выдать ровно это.

## Шаг 1 — класс `Value`

Оборачиваем число в объект, который помнит своих родителей и функцию
`_backward`. Граф строится сам, пока мы пишем `a*b + c`.

Обрати внимание на `+=` в `_backward` (не `=`) — вклады должны складываться.

In [ ]:
import math

class Value:
    def __init__(self, data, _children=()):
        self.data = data
        self.grad = 0.0                  # пока не звали backward — градиент 0
        self._backward = lambda: None    # как протолкнуть градиент детям
        self._prev = set(_children)      # родители в графе

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other))
        def _backward():
            self.grad  += 1.0 * out.grad   # + пропускает градиент насквозь
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other))
        def _backward():
            self.grad  += other.data * out.grad   # × отдаёт другой множитель
            other.grad += self.data  * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,))
        def _backward():
            self.grad += (1 - t**2) * out.grad     # производная tanh
        out._backward = _backward
        return out

    def backward(self):
        # сначала посчитать узел, потом его детей → топологический порядок
        topo, visited = [], set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)
        build(self)
        self.grad = 1.0                  # dL/dL = 1, отсюда стартует всё
        for v in reversed(topo):
            v._backward()

## Шаг 2 — код подтверждает то, что ты посчитал руками

In [ ]:
a = Value(2.0)
b = Value(3.0)
c = Value(4.0)

e = a * b
L = e + c

L.backward()

print("L      =", L.data)     # 10.0
print("a.grad =", a.grad)     # 3.0  <- как ты посчитал руками
print("b.grad =", b.grad)     # 2.0
print("c.grad =", c.grad)     # 1.0

Совпало? Поздравляю — ты только что написал `.backward()`.

## Шаг 3 — обучаем один нейрон

Нейрон: `y = tanh(w*x + b)`. Хотим, чтобы при `x=1` он выдавал `-1`.
`loss` собираем тоже из `Value`, поэтому `loss.backward()` сам дотечёт до `w` и `b`.

In [ ]:
x = Value(1.0)
w = Value(0.8)
b = Value(0.1)
target = -1.0

for step in range(20):
    # forward
    y    = (w * x + b).tanh()
    diff = y + Value(-target)     # y - target
    loss = diff * diff            # квадрат ошибки

    # backward
    w.grad = 0.0                  # ОБНУЛИТЬ перед каждым шагом!
    b.grad = 0.0
    loss.backward()

    # шаг градиентного спуска: против градиента
    w.data -= 0.1 * w.grad
    b.data -= 0.1 * b.grad
    print(f"step {step:2d}  y={y.data:+.3f}  loss={loss.data:.4f}")

`loss` ползёт к нулю, `y` — к `-1`. Это тот же цикл, что обучает любую LLM:
`forward -> loss -> backward -> шаг`.

---
# Домашка

## ДЗ-1. Добавь операцию `__pow__`

Нужна, чтобы считать loss как `(y - target)**2` прямо через `Value`.
Локальная производная: `d(x**n)/dx = n * x**(n-1)`.

Допиши метод ниже и **добавь его в класс `Value`** (можно переопределить
класс целиком в этой ячейке, скопировав Шаг 1 и вставив `__pow__`).

In [ ]:
# TODO: реализуй возведение в степень.
# Вставь этот метод внутрь класса Value (рядом с __add__ / __mul__).

def __pow__(self, n):            # n — обычное число (int/float)
    assert isinstance(n, (int, float))
    out = Value(self.data ** n, (self,))
    def _backward():
        # TODO: self.grad += (локальная производная) * out.grad
        # подсказка: d(x**n)/dx = n * x**(n-1)
        self.grad += None  # <-- замени None
    out._backward = _backward
    return out

## ДЗ-2. Перепиши цикл так, чтобы loss был `Value`

Теперь loss можно записать через `**2`, а градиенты текут сами.
Заверни обучение в функцию, чтобы переиспользовать её для разных целей.

Шаг обучения здесь `lr=0.3` (не `0.1` как в демо): цель `-1` — самая
медленная, потому что `tanh` у краёв «насыщается» и градиент почти
исчезает. Шаг побольше помогает уложиться в 20 шагов под `0.01`. Это
первое знакомство с **learning rate** — вернёмся к нему в модуле 5.

In [ ]:
def train_neuron(target, steps=20, lr=0.3):
    x = Value(1.0)
    w = Value(0.8)
    b = Value(0.1)
    losses = []
    for step in range(steps):
        # forward
        y    = (w * x + b).tanh()
        # TODO: loss = (y - target) ** 2, собранный из Value
        #       подсказка: y + Value(-target), затем ** 2
        loss = None  # <-- замени

        # backward
        w.grad = 0.0
        b.grad = 0.0
        loss.backward()

        # шаг
        w.data -= lr * w.grad
        b.data -= lr * b.grad
        losses.append(loss.data)
    return losses

## ДЗ-3. Обучи на нескольких целях и построй график loss

In [ ]:
import matplotlib.pyplot as plt

for target in [-1.0, 0.5, 0.9]:
    losses = train_neuron(target)
    plt.plot(losses, label=f"target={target}")

plt.xlabel("шаг")
plt.ylabel("loss")
plt.title("Падение loss по шагам")
plt.legend()
plt.show()

## ДЗ-4. Что будет без обнуления градиентов?

Убери строки `w.grad = 0.0` / `b.grad = 0.0` в `train_neuron`, перезапусти
и понаблюдай. **Проверь экспериментом**, не угадывай.

Подсказка: следи не только за loss (в этом игрушечном примере он может даже
упасть *быстрее*!), но и за `w.grad`. Распечатай его на каждом шаге. Запиши
абзац: чем `w.grad` без обнуления отличается от наклона в текущей точке и
почему (подсказка: в `_backward` стоит `+=`, поэтому вклады суммируются).

In [ ]:
# TODO: копия train_neuron без строк w.grad=0 / b.grad=0.
#       На каждом шаге печатай w.grad — увидишь, как он копит сумму.
#       Ниже добавь markdown-ячейку с объяснением в один абзац.

## Что сдать

- Публичная ссылка на этот Colab-ноутбук (запускается сверху вниз без правок).
- График падения loss хотя бы для одной цели.
- Абзац про эксперимент без обнуления градиентов.

**Критерий приёма:** `loss.backward()` сам заполняет `w.grad`/`b.grad`,
loss за 20 шагов уходит ниже `0.01`.

В чат как `[Модуль 4a, ДЗ] {ссылка}`.